# Cerebra / TRIBE v2 — GPU inference

Use a **T4 GPU** runtime: `Runtime → Change runtime type → T4 GPU`. Add your Hugging Face read token to Colab Secrets as `HF_TOKEN`.

## 1. Install packages
Run once per Colab runtime. If you already saw a NumPy import error, use `Runtime → Restart session` first, then run this section before any other code.

In [ ]:
import subprocess, sys, sysconfig, shutil
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

def log(message):
    print(f'[{datetime.now():%H:%M:%S}] {message}', flush=True)

def pip_install(*packages, force_reinstall=False, no_deps=False):
    command = [sys.executable, '-m', 'pip', 'install', '--progress-bar', 'on', '--no-cache-dir']
    if force_reinstall:
        command.append('--force-reinstall')
    if no_deps:
        command.append('--no-deps')
    command.extend(packages)
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f'pip failed with exit code {result.returncode}. The package output above contains the cause.')

install_bar = tqdm(total=2, desc='Installing TRIBE v2 packages', unit='step')
# TRIBE v2 explicitly requires NumPy 2.2.6. Colab can retain old .py files
# beside new compiled extensions, so remove every NumPy install artifact first.
log('Removing stale NumPy files before a clean ABI install…')
site_packages = Path(sysconfig.get_paths()['purelib'])
for target in [site_packages / 'numpy', site_packages / 'numpy.libs', *site_packages.glob('numpy-*.dist-info')]:
    if target.exists():
        shutil.rmtree(target)
for module_name in list(sys.modules):
    if module_name == 'numpy' or module_name.startswith('numpy.'):
        del sys.modules[module_name]
log('Installing clean NumPy 2.2.6…')
pip_install('numpy==2.2.6', no_deps=True)
install_bar.update(1)
log('Installing TRIBE v2 and its feature-extraction dependencies…')
pip_install('git+https://github.com/facebookresearch/tribev2.git')
check = subprocess.run([sys.executable, '-c', 'import numpy; import tribev2; print(numpy.__version__)'], text=True, capture_output=True)
if check.returncode:
    raise RuntimeError(f'Package verification failed:\nSTDOUT:\n{check.stdout}\nSTDERR:\n{check.stderr}')
log(f'Verified NumPy {check.stdout.strip()} and tribev2 imports.')
install_bar.update(1); install_bar.close()
log('TRIBE v2 dependencies installed.')


## 2. Put a video in `downloads/`
This creates `/content/downloads`. Upload a video into it, then set `VIDEO_NAME` below if there is more than one.

In [ ]:
from pathlib import Path
from google.colab import files
from tqdm.auto import tqdm

INPUT_DIR = Path('/content/downloads')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.webm', '.avi', '.mkv'}

# Leave None to use the only video in downloads/. Set an exact filename when needed.
VIDEO_NAME = None

input_bar = tqdm(total=2, desc='Preparing input video', unit='step')
log(f'Checking {INPUT_DIR}…')
existing = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS]
input_bar.update(1)
if not existing:
    print('Upload a video. It will be saved in /content/downloads/.')
    for name, data in files.upload().items():
        (INPUT_DIR / name).write_bytes(data)
    existing = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS]
input_bar.update(1); input_bar.close()

log('Available videos:')
for path in existing:
    print(' -', path.name)


## 3. Run TRIBE v2 on GPU
This fails fast if Colab is not using CUDA—no hidden multi-hour CPU run.

In [ ]:
import os
import torch
from tqdm.auto import tqdm
from google.colab import userdata
from tribev2 import TribeModel

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if not hf_token:
    raise RuntimeError('Missing HF_TOKEN. Add it in Colab Secrets, then rerun this cell.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a T4 GPU runtime before running TRIBE v2.')

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

videos = sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS)
if VIDEO_NAME:
    video_path = INPUT_DIR / VIDEO_NAME
elif len(videos) == 1:
    video_path = videos[0]
else:
    raise RuntimeError('Set VIDEO_NAME in section 2 to one of the listed videos.')
if not video_path.exists():
    raise FileNotFoundError(video_path)

CACHE_DIR = Path('/content/tribev2-cache')
CACHE_DIR.mkdir(exist_ok=True)
run_bar = tqdm(total=3, desc='TRIBE v2 inference', unit='stage')
log(f'GPU: {torch.cuda.get_device_name(0)}')
log(f'Video: {video_path.name}')

log('Loading TRIBE v2 weights and feature extractors…')
model = TribeModel.from_pretrained(
    'facebook/tribev2',
    cache_folder=CACHE_DIR,
    device='cuda',
)
run_bar.update(1)
log('Extracting video, audio, and language events…')
events = model.get_events_dataframe(video_path=str(video_path))
run_bar.update(1)
log('Predicting cortical responses (the per-frame progress bar below is TRIBE itself)…')
predictions, _ = model.predict(events, verbose=True)
run_bar.update(1); run_bar.close()
log(f'Inference complete. Prediction shape: {predictions.shape}')


## 4. Export results
Exports a clean, video-named folder and a downloadable ZIP.

In [ ]:
import json, re, shutil
import numpy as np
from google.colab import files
from tqdm.auto import tqdm

safe_stem = re.sub(r'[^A-Za-z0-9._-]+', '_', video_path.stem).strip('_')
result_dir = Path('/content/outputs') / f'{safe_stem}_tribev2'
result_dir.mkdir(parents=True, exist_ok=True)

export_bar = tqdm(total=3, desc='Exporting results', unit='file')
# One portable results file: full cortical predictions plus timing metadata.
result_file = result_dir / f'{safe_stem}_tribev2_results.npz'
np.savez_compressed(result_file,
                    predictions=predictions.astype(np.float32),
                    tr_seconds=np.float32(model.data.TR),
                    video_name=np.array(video_path.name))
export_bar.update(1); log(f'Wrote {result_file.name}')
metadata = {
    'video_name': video_path.name,
    'prediction_shape': list(predictions.shape),
    'tr_seconds': float(model.data.TR),
    'duration_seconds': float(predictions.shape[0] * model.data.TR),
    'device': torch.cuda.get_device_name(0),
    'results_file': result_file.name,
}
metadata_file = result_dir / f'{safe_stem}_tribev2_metadata.json'
metadata_file.write_text(json.dumps(metadata, indent=2))
export_bar.update(1); log(f'Wrote {metadata_file.name}')

archive = shutil.make_archive(str(result_dir), 'zip', result_dir)
export_bar.update(1); export_bar.close()
log(f'Created {Path(archive).name}')
print(json.dumps(metadata, indent=2))
log(f'Results folder: {result_dir}')
files.download(archive)
